# Project: Learning Japanese with an MLP

---

Prof. Dr.-Ing. Antje Muntzinger, Hochschule für Technik Stuttgart

antje.muntzinger@hft-stuttgart.de

---

In this project, you will apply your knowledge to the image dataset **Kuzushiji-MNIST (KMNIST)**, consisting of 28×28 grayscale images of historical Japanese cursive characters in ten classes. You do not need to read Japanese; we use class indices 0–9. We will build a feedforward network, train it, compare versions with and without dropout, and inspect its predictions. 

## 1. Load and inspect the data

We start with importing the necessary libraries and loading our Japanese digit dataset. 

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

SEED = 11
torch.manual_seed(SEED)
DATA_ROOT = './data'
DOWNLOAD = True  # False if the complete raw dataset is already available locally


We use a fixed, class-stratified subset to keep training manageable on a CPU: 10,000 training images and 2,000 validation images. Both come from the official training set; the remaining training images are unused in this introductory project. Keep the official test set for the final evaluation.

We convert the selected images into tensors of shape `(N,1,28,28)` and divide pixel values by 255. This is a fixed transformation, so it does not estimate statistics from validation or test data. Labels are integer class indices with dtype `torch.long`.


In [ ]:
trainset_full = datasets.KMNIST(root=DATA_ROOT, train=True, download=DOWNLOAD,
                                transform=transforms.ToTensor())
all_labels = trainset_full.targets.numpy()
all_indices = np.arange(len(trainset_full))
selected_idx, _ = train_test_split(all_indices, train_size=12000,
                                    stratify=all_labels, random_state=SEED)
train_idx, val_idx = train_test_split(selected_idx, test_size=2000,
                                      stratify=all_labels[selected_idx], random_state=SEED)

# Using the raw tensors avoids repeatedly converting PIL images during training.
x_train = trainset_full.data[train_idx].float().unsqueeze(1) / 255.0
y_train = trainset_full.targets[train_idx].long()
x_val = trainset_full.data[val_idx].float().unsqueeze(1) / 255.0
y_val = trainset_full.targets[val_idx].long()

def make_trainloader(seed=SEED):
    return DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)

valloader = DataLoader(TensorDataset(x_val, y_val), batch_size=256, num_workers=0)
print('Training / validation samples:', len(y_train), len(y_val))
print('Training samples per class:', torch.bincount(y_train).tolist())


**TODO 1:** Show eight training images with their class indices. Print the shape of an image batch and its flattened version. How many input values does the network need per image? **(4 points)**

**HINT:** Use `image[0]` for plotting a grayscale image and `inputs.flatten(start_dim=1)` for flattening a batch. Keep the batch dimension.


In [ ]:
#YOUR CODE HERE


**ANSWER:** 


## 2. Build the network

**TODO 2a:** What is dropout and why is it used in neural networks? How does the dropout rate `p` affect the model's performance? Do some research and provide a brief explanation. **(2 points)**

**ANSWER:** 


**TODO 2b:** Define a class `Classifier` with three fully-connected layers: 784 → 128 → 64 → 10. Use ReLU after each hidden layer, flatten the input in `forward`, and return raw logits. Add `nn.Dropout(p)` after each hidden activation. The classifier class should take the dropout probability `p` as input and its default probability should be `p=0.0`. Instanciate a classifier named `model`. **(6 points)**


In [ ]:
#YOUR CODE HERE



#END STUDENT CODE

print(model)
print('Output shape:', model(inputs).shape)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))


**TODO 2c:** Which loss function matches this output? Why should you not apply softmax before calculating the loss? Calculate the number of trainable parameters (write down your calculation steps) and check your result above. **(4 points)**

**ANSWER:** 


## 3. Train and validate the network

First, we will write the complete training loop in one place. The outer loop runs over **epochs**; inside it, the training loop visits all mini-batches and updates the weights. A separate validation loop then checks the model without updating it.

The cell below includes everything: a fresh model, loss function, optimizer, both loops, loss tracking, and checkpoint selection. Read it from top to bottom and follow which operations happen once, once per epoch, or once per batch.

We keep the checkpoint with the lowest validation loss. `copy.deepcopy(model.state_dict())` saves an independent copy of its parameters. At the end, we restore that checkpoint. The full epoch budget is still completed; this is checkpoint selection rather than early stopping.


**TODO 3a:** Complete the five central steps inside the training mini-batch loop: clear the gradients, calculate the logits, calculate the loss, backpropagate, and update the parameters. All surrounding code is provided. **(3 points)**


In [ ]:
# Start a new experiment with a fresh model and optimizer.
torch.manual_seed(SEED)
model = Classifier(p=0.0)
criterion = nn.CrossEntropyLoss()
LEARNING_RATE = 0.001
EPOCHS = 12
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
trainloader = make_trainloader(seed=SEED)

train_losses, val_losses, val_accuracies = [], [], []
best_val_loss = float('inf')
best_state = None
best_epoch = None

for epoch in range(EPOCHS):
    # --- Training: update the weights using all training mini-batches. ---
    model.train()
    train_loss_sum = 0.0
    for inputs, targets in trainloader:
        
        #YOUR CODE HERE



        #END STUDENT CODE

        # Weight each batch by its size, including the smaller last batch.
        train_loss_sum += loss.item() * len(inputs)
    train_loss = train_loss_sum / len(trainloader.dataset)

    # --- Validation: inspect the model without changing its weights. ---
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    with torch.no_grad():
        for inputs, targets in valloader:
            logits = model(inputs)
            loss = criterion(logits, targets)
            val_loss_sum += loss.item() * len(inputs)
            val_correct += (logits.argmax(dim=1) == targets).sum().item()
    val_loss = val_loss_sum / len(valloader.dataset)
    val_accuracy = val_correct / len(valloader.dataset)

    # Record the learning curves after each epoch.
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    # Keep an independent copy of the best validation checkpoint.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

    if (epoch + 1) % 4 == 0:
        print(f'Epoch {epoch+1:2d}: train loss {train_loss:.4f}, '
              f'val loss {val_loss:.4f}, val accuracy {val_accuracy:.1%}')

# Use the selected checkpoint, rather than automatically keeping the last one.
model.load_state_dict(best_state)
print('Selected epoch:', best_epoch)


**TODO 3b:** Explain what `zero_grad()`, `model.train()`, `model.eval()`, and `torch.no_grad()` do. Which of the code lines above change model parameters, and which lines only inspect or record results? **(4 points)**


**ANSWER:** 


**TODO 3c:** Plot the recorded training and validation loss.  **(2 points)**

**HINT:** Use `train_losses` and `val_losses`, with epoch numbers from 1 to `EPOCHS` on the horizontal axis.


In [ ]:
#YOUR CODE HERE


## 4. Compare with dropout

Now that the complete loop is visible, we can package the same steps into functions and reuse them for another experiment. The functions below are provided:

* `train_epoch` contains the training mini-batch loop for one epoch.
* `evaluate` contains the validation loop and can also be used on test data.
* `fit` contains model-training setup, the outer epoch loop, loss tracking, and checkpoint selection.

Compare these functions with Section 3: the learning algorithm stays the same; only the organization of the code changes. Each call to `fit` gets a fresh optimizer and a training loader with a reset batch-order seed. The caller supplies a fresh model.


In [ ]:
# The same mini-batch training steps as in Section 3, now in a reusable function.
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(inputs)
    return total_loss / len(loader.dataset)

# The validation steps from Section 3, reused for validation or testing.
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            logits = model(inputs)
            loss = criterion(logits, targets)
            total_loss += loss.item() * len(inputs)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
    return total_loss / len(loader.dataset), total_correct / len(loader.dataset)

# The outer epoch loop and checkpoint selection from Section 3.
def fit(model, epochs=EPOCHS, lr=LEARNING_RATE, seed=SEED):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    trainloader = make_trainloader(seed=seed)
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    best_loss, best_state, best_epoch = float('inf'), None, None
    for epoch in range(epochs):
        train_loss = train_epoch(model, trainloader, criterion, optimizer)
        val_loss, val_accuracy = evaluate(model, valloader, criterion)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
        if (epoch + 1) % 4 == 0:
            print(f'Epoch {epoch+1:2d}: train loss {train_loss:.4f}, '
                  f'val loss {val_loss:.4f}, val accuracy {val_accuracy:.1%}')
    model.load_state_dict(best_state)
    return history, best_epoch


The trained model from Section 3 is our comparison without dropout. We keep that model and its recorded curves below; it does not need to be trained again. The dictionaries simply associate each experiment name with its model or recorded results.


In [ ]:
models = {'No dropout': model}
histories = {'No dropout': {'train_loss': train_losses,
                            'val_loss': val_losses,
                            'val_accuracy': val_accuracies}}


**TODO 4:** Train a fresh network with `p=0.2`, using `fit` above. Keep the architecture, data split, initialization seed, batch order seed, learning rate, and epoch budget the same. Add its model and history to the dictionaries. Plot both sets of learning curves and compare the restored checkpoints using validation loss and accuracy. Does dropout help here? **(6 points)**


In [ ]:
#YOUR CODE HERE



**ANSWER:**


## 5. Final test and predictions 

Finally, we select the model with the lowest validation loss and then evaluate it on the official test set. 

In [ ]:
# Load the official test set only after completing the model comparison.
testset = datasets.KMNIST(root=DATA_ROOT, train=False, download=DOWNLOAD,
                          transform=transforms.ToTensor())
x_test = testset.data.float().unsqueeze(1) / 255.0
y_test = testset.targets.long()
testloader = DataLoader(TensorDataset(x_test, y_test), batch_size=256, num_workers=0)
print('Official test samples:', len(y_test))


In [ ]:
selected_name = min(models, key=lambda name:
                    evaluate(models[name], valloader, nn.CrossEntropyLoss())[0])
selected_model = models[selected_name]
test_loss, test_accuracy = evaluate(selected_model, testloader, nn.CrossEntropyLoss())
print(f'Selected: {selected_name}; test loss {test_loss:.4f}, test accuracy {test_accuracy:.1%}')
selected_model.eval()
with torch.no_grad():
    test_logits = selected_model(x_test)
    predictions = test_logits.argmax(dim=1)
ConfusionMatrixDisplay.from_predictions(y_test.numpy(), predictions.numpy(),
                                         labels=list(range(10)), cmap='Blues')
plt.title('Final test: true vs. predicted class'); plt.show()

We calculate and print the class probabilities for one test image, and show some misclassified images with their true and predicted class:

In [ ]:
probabilities = torch.softmax(test_logits[:1], dim=1)
print('First image: true class', y_test[0].item(), ', predicted class', predictions[0].item())
print('Class probabilities:', probabilities[0])
assert torch.allclose(probabilities.sum(dim=1), torch.ones(1))
wrong = (predictions != y_test).nonzero(as_tuple=True)[0][:6]
if len(wrong) == 0:
    print('No misclassified test images.')
else:
    fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2), squeeze=False)
    for ax, idx in zip(axes[0], wrong):
        ax.imshow(x_test[idx, 0], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'True {y_test[idx].item()} / Pred {predictions[idx].item()}')
        ax.axis('off')
    plt.tight_layout(); 
    plt.show()


## 6. Discussion 

**TODO 6:** In a short paragraph, answer the following questions: Does the network generalize? Which classes are frequently confused? Name one limitation of your evaluation. **(3 points)**

**ANSWER:** 